Ingestion layer that accepts PDFs, images, audio, and video files, converts each format into text using extraction, OCR, or transcription, then chunks the text, generates embeddings, and stores everything in a UUID-based filesystem folder for future retrieval.

In [12]:
from pathlib import Path

import fitz
import whisper
import pytesseract

from PIL import Image
from moviepy.video.io.VideoFileClip import VideoFileClip

BASE_INPUT_DIR = Path("data/inputs/test_inputs")

PDF_PATH = BASE_INPUT_DIR / "deploy_python.pdf"
IMAGE_PATH = BASE_INPUT_DIR / "deploy_python.png"
AUDIO_PATH = BASE_INPUT_DIR / "deploy_python.mp3"
VIDEO_PATH = BASE_INPUT_DIR / "deploy_python.mp4"

### Extract text from PDF

In [13]:
def extract_text_from_pdf(pdf_path):
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    extracted_pages = []

    pdf_doc = fitz.open(pdf_path)

    for page_number, page in enumerate(pdf_doc, start=1):
        page_text = page.get_text()

        if page_text.strip():
            extracted_pages.append(f"[PDF Page {page_number}]\n{page_text}")

    pdf_doc.close()

    return "\n\n".join(extracted_pages)

### Extract text from image using OCR

In [14]:
def extract_text_from_image(image_path):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = Image.open(image_path)
    image_text = pytesseract.image_to_string(image)

    return f"[Image Text]\n{image_text}"

### Extract transcript from audio

In [15]:
def extract_text_from_audio(audio_path, model_size="base"):
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    whisper_model = whisper.load_model(model_size)
    result = whisper_model.transcribe(str(audio_path))

    transcript = result.get("text", "")

    return f"[Audio Transcript]\n{transcript}"

### Extract transcript from video

In [16]:
def extract_text_from_video(video_path, output_audio_path="data/inputs/test_inputs/video_audio.wav", model_size="base"):
    video_path = Path(video_path)
    output_audio_path = Path(output_audio_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    output_audio_path.parent.mkdir(parents=True, exist_ok=True)

    video_clip = VideoFileClip(str(video_path))

    if video_clip.audio is None:
        video_clip.close()
        raise ValueError("This video has no audio track.")

    video_clip.audio.write_audiofile(
        str(output_audio_path),
        logger=None
    )

    video_clip.close()

    whisper_model = whisper.load_model(model_size)
    result = whisper_model.transcribe(str(output_audio_path))

    transcript = result.get("text", "")

    return f"[Video Transcript]\n{transcript}"

### Extracted & Saving 

In [17]:
import json
import uuid
from pathlib import Path


OUTPUT_DIR = Path("data/outputs")


def save_extracted_text_with_uuid(extracted_data):
    job_id = str(uuid.uuid4())
    job_folder = OUTPUT_DIR / job_id
    job_folder.mkdir(parents=True, exist_ok=True)

    combined_text_parts = []

    for source_name, source_text in extracted_data.items():
        file_path = job_folder / f"{source_name}_text.txt"

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(source_text)

        combined_text_parts.append(source_text)

    combined_text = "\n\n".join(combined_text_parts)

    with open(job_folder / "all_extracted_text.txt", "w", encoding="utf-8") as f:
        f.write(combined_text)

    metadata = {
        "job_id": job_id,
        "sources": list(extracted_data.keys()),
        "output_folder": str(job_folder)
    }

    with open(job_folder / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "job_id": job_id,
        "output_folder": str(job_folder),
        "saved_files": [str(p) for p in job_folder.iterdir()]
    }

### Testing

In [18]:
def extract_all_test_inputs():
    extracted_data = {}

    extracted_data["pdf"] = extract_text_from_pdf(PDF_PATH)
    extracted_data["image"] = extract_text_from_image(IMAGE_PATH)
    extracted_data["audio"] = extract_text_from_audio(AUDIO_PATH)
    extracted_data["video"] = extract_text_from_video(VIDEO_PATH)

    return extracted_data


all_text_data = extract_all_test_inputs()

save_result = save_extracted_text_with_uuid(all_text_data)

save_result

/Users/jigglyyy/Documents/projects/nlp/.venv/lib/python3.13/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/jigglyyy/Documents/projects/nlp/.venv/lib/python3.13/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


{'job_id': '71a363ee-27b0-4abc-b0f0-f078fe711fc4',
 'output_folder': 'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4',
 'saved_files': ['data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/video_text.txt',
  'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/pdf_text.txt',
  'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/image_text.txt',
  'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/metadata.json',
  'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/audio_text.txt',
  'data/outputs/71a363ee-27b0-4abc-b0f0-f078fe711fc4/all_extracted_text.txt']}